In [7]:
import json
from pymorphy2 import MorphAnalyzer
from tqdm import tqdm
import os
import pandas as pd

In [8]:
def preprocess(text):
    '''Лемматизация'''
    morph = MorphAnalyzer()
    lemmas = []
    for word in text.split(' '):
        word = word.strip('!@#$%^&*()_+-=?><.,\'\":;][\{\}]`~\n\t\s—»«').lower()
        ana = morph.parse(word)
        lemmas.append(ana[0].normal_form)
    return lemmas

In [12]:
with open('../../data/test.json', 'r', encoding='utf-8') as f:
    test_file = json.load(f)
test_video_names = set()
for item in test_file:
    test_video_names.add(item['video'])

In [20]:
test_video_names

{'60-ти летие фотографа Прикащикова. С субтитрами.mp4',
 'VIII Всероссийский молодежный форум ВОГ. День третий. «ВОГ и я».mp4',
 'VIII Всероссийский молодежный форум ВОГ. День шестой, заключительный..mp4',
 'Американский глухой художник Гренвилль Редмонд. С субтитрами.mp4',
 'Ваша безопасность в ваших руках. С субтитрами.mp4',
 'Доверяй, но проверяй. С субтитрами.mp4',
 'Жизнь и приключения глухих в 1950-е годы. Интервью с Ниной Марийсовой. 2 часть. С субтитрами.mp4',
 'И вновь о переводчиках. С субтитрами.mp4',
 'И.Б. Дубовицкий. История ВОГ в лицах. С субтитрами.mp4',
 'И.И. Снетков. История ВОГ в лицах. С субтитрами.mp4',
 'Интервью с глухим архитектором Юрием Соколовым. 3 часть. С субтитрами.mp4',
 'Интервью с новым директором Санаторий МАЯК Сергеем Мазуром. С субтитрами.mp4',
 'Интервью с серебряным призером 1965 года Н.Сусловым. С субтитрами.mp4',
 'Лекция Паленного в музее Гараж. С субтитрами.mp4',
 'Международный день жестовых языков. С субтитрами.mp4',
 'Мониторинг образования

In [31]:
sub_path = '../load_data/id2name.json'
id2name = json.load(open(sub_path, 'r', encoding='utf-8'))
bad_symbols = '?\":/|'
sub_files = os.listdir("SLR Project_subs/")

captions = [] # Текст субтитров
tokenized_captions = [] # Лемматизированные субтитры
video_info = [] # video_name, start, end

for caption_file in tqdm(sub_files):

    name = caption_file.split('/')[-1].split('.')[0]
    video_name = id2name[name]+'.mp4'

    if video_name in test_video_names:
        for bs in bad_symbols:
            video_name = video_name.replace(bs, '')
        if video_name == 'Интервью с двукратным сурдлимпийским чемпионом Владиславом Винником. 2 часть.mp4':
            video_name = "Интервью с двукратным сурдлимпийским чемпионом Владиславом Винником. 2 часть. С субтитрами.mp4"
        fn = 'SLR Project_subs/'+ caption_file.split('/')[-1]
        with open(fn, 'r', encoding='utf-8') as f:
            for caption in json.load(f):
                cap_dict = caption.copy()
                if type(cap_dict['text']) == str:
                    captions.append(cap_dict['text'])
                    tokenized_captions.append(preprocess(cap_dict['text']))
                    video_info.append([video_name, cap_dict['start'], cap_dict['start'] + cap_dict['duration']])

100%|██████████| 321/321 [11:22<00:00,  2.13s/it]


In [33]:
with open('../../data/test_captions.json', 'w', encoding='utf-8') as f:
    json.dump([captions, tokenized_captions, video_info], f, ensure_ascii=False, indent=4)